# BPR (H3) - con protocolo comun `recsys_protocol`

Notebook de **H3** (carpeta `H3/`). Version de BPR/ALS/Most-Popular que adopta
`recsys_protocol.py`. **No toca H2:** el original queda intacto en `../H2/BPR.ipynb`.

**Cambios vs H2** (para *aislar* el efecto del protocolo):
- `set_global_seed()` (FIX #3) + muestreo `sample_users()` determinista e
  independiente del orden (FIX #2): el eval pool es **el mismo que en `SBERT.ipynb`**.
- k-core, split (`build_splits`) y metricas vienen del **modulo** (unica fuente
  de verdad), verificados identicos a H2.
- **Novelty estandarizada** (denominador = nro interacciones positivas en train,
  ignora no vistos) y **catalogo = items en train** -> misma convencion que el
  resto de modelos. Ojo: H2 BPR usaba otra escala de Novelty (nro usuarios) y
  otro catalogo (10% muestreado), asi que Novelty/Coverage no son directamente
  comparables con H2 (NDCG/Recall si). La **ILD** de este notebook usa el espacio latente del modelo (no comparable entre modelos); la **ILD semantica unificada** y la tabla cross-modelo con **media +/- desv (multi-seed)** estan en `SBERT.ipynb` (seccion 11).

**Como comparar:** correr en Colab (CPU + High-RAM) y contrastar con `SBERT.ipynb`
sobre el MISMO eval pool. Referencia H2 NDCG@10: ALS 0.0307 - BPR 0.0298 - MostPop 0.0171.

**Setup:** la primera celda clona el repo y carga el modulo desde `H3/`.


# Entrega H2 — Proyecto RecSys
## Tema: Multimodalidad — Recomendación de Videojuegos

**Dataset:** [Game Recommendations on Steam](https://www.kaggle.com/datasets/antonkozyriev/game-recommendations-on-steam)

En H1 se establecieron cuatro baselines: Random, Most Popular, CB-TF-IDF y ALS.  
En H2 se incorpora **BPR (Bayesian Personalized Ranking)** como nuevo modelo colaborativo que optimiza ranking pairwise directamente, más alineado con la tarea top-K que ALS.

La evaluación se extiende a K ∈ {5, 10, 20} e incluye métricas de diversidad: Coverage, Novelty e ILD (Intra-List Diversity).

In [ ]:
!pip install -q kagglehub implicit

import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import normalize
from scipy.sparse import csr_matrix
from IPython.display import display
import implicit
import kagglehub

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

# ── Protocolo experimental comun: lo trae el repo de la entrega en H3/ ──
# Intenta primero una copia local (subida a mano o repo ya clonado en esta
# sesion); si no esta, CLONA el repo y usa H3/recsys_protocol.py. Asi no hay que
# subir el archivo a mano en cada cuadernillo nuevo: basta con esta celda.
import glob, subprocess
REPO_URL = 'https://github.com/Benjaa7/Proyecto-RecSys.git'   # <-- ajusta si tu repo cambia
REPO_DIR = '/content/Proyecto-RecSys'

def _locate_protocol():
    # Repo PRIMERO: clonar si falta; si ya esta clonado, traer la ULTIMA version
    # (fetch + reset --hard). Esto evita el bug de usar un clon viejo cacheado de
    # una sesion anterior (un 'pull' tras encontrar copia local no se ejecutaba).
    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        subprocess.run(['git', '-C', REPO_DIR, 'fetch', '-q', '--depth', '1', 'origin'], check=False)
        subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', '-q', 'FETCH_HEAD'], check=False)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', '-q', REPO_URL, REPO_DIR], check=False)
    h3 = os.path.join(REPO_DIR, 'H3')
    if os.path.exists(os.path.join(h3, 'recsys_protocol.py')):
        return h3
    # Fallback: copia local suelta (subida a mano), solo si el repo no esta disponible.
    for c in ['/content', '.', '..'] + sorted(glob.glob('/content/*')) + sorted(glob.glob('/content/drive/MyDrive/*')):
        if c and os.path.exists(os.path.join(c, 'recsys_protocol.py')):
            return c
    return None

_root = _locate_protocol()
if _root is None:
    raise ModuleNotFoundError(
        "No encuentro recsys_protocol.py. Confirma que esta commiteado en "
        f"{REPO_URL} bajo H3/, o sube el archivo a /content.")
sys.path.insert(0, _root)
# Re-import fresco: si el modulo ya estaba en memoria (re-correr la celda sin
# reiniciar el runtime), descartarlo para tomar la version recien traida.
for _m in [m for m in list(sys.modules) if m == 'recsys_protocol' or m.startswith('recsys_protocol.')]:
    del sys.modules[_m]
print('recsys_protocol desde:', _root)

from recsys_protocol import (
    SEED, ProtocolConfig, set_global_seed, sample_users, iterative_k_core,
    build_splits, positives, popularity_counts, most_popular_list, reproducibility_note,
    # harness de metricas (unica fuente de verdad; identico a H2 para este pipeline)
    evaluate_at_ks, coverage_at_k, novelty_at_k, ild_at_k, activity_ndcg,
)

set_global_seed()  # FIX #3: seed unica de proyecto (numpy/random/torch/cuda)

# frac_train == frac_eval == 0.10 reproduce el regimen de H2: el UNICO cambio
# respecto al BPR original es el muestreador determinista (FIX #2).
cfg = ProtocolConfig(frac_eval=0.10, frac_train=0.10, min_user=5, min_game=20,
                     max_eval_users=2000, ks=(5, 10, 20),
                     user_col='user_id', item_col='app_id', time_col='date',
                     pos_col='is_recommended')

# Alias para que el resto del notebook (celdas sin tocar) siga funcionando igual
RANDOM_STATE = SEED
KS           = tuple(cfg.ks)
TOP_K        = max(cfg.ks)
MAX_EVAL     = cfg.max_eval_users
ALPHA        = 40
MIN_USER     = cfg.min_user
MIN_GAME     = cfg.min_game
SAMPLE_FRAC  = cfg.frac_eval

print(reproducibility_note(cfg, 'Kozyriev (BPR piloto)'))


## 1. Carga de datos

Se carga `recommendations.csv` (~41 M filas) en chunks con tipos de dato optimizados para reducir uso de memoria.

In [ ]:
PATH = kagglehub.dataset_download('antonkozyriev/game-recommendations-on-steam')
print('Archivos:', os.listdir(PATH))

chunks = []
for chunk in pd.read_csv(
    os.path.join(PATH, 'recommendations.csv'),
    chunksize=1_000_000,
    dtype={'app_id': 'int32', 'user_id': 'int32',
           'hours': 'float32', 'is_recommended': 'bool',
           'helpful': 'int16', 'funny': 'int16'},
    parse_dates=['date']
):
    chunks.append(chunk)
recs = pd.concat(chunks, ignore_index=True)
print(f'Cargado: {len(recs):,} filas — {recs.memory_usage(deep=True).sum()/1e9:.2f} GB')

## 2. Preprocesamiento

Se replican exactamente los pasos del H1:
1. **Deduplicación** — por par (user, juego), manteniendo la interacción más reciente.
2. **Señal de confianza** — `confidence = log1p(hours)` para interacciones positivas.
3. **K-core iterativo** — `min_user=5`, `min_game=20`; converge en 9 iteraciones.
4. **Subsample 10%** de usuarios para tractabilidad.
5. **Split temporal leave-one-out** — última interacción de cada usuario como test; solo se evalúan los usuarios cuyo ítem de test es positivo.

In [ ]:
# 1. Deduplicacion (idem H1/H2: por par user-juego, interaccion mas reciente)
recs = (recs.sort_values('date')
            .drop_duplicates(subset=['user_id', 'app_id'], keep='last')
            .reset_index(drop=True))
print(f'Post-dedup: {len(recs):,}')

# 2. Senal de confianza
recs['confidence'] = np.where(
    recs['is_recommended'],
    np.log1p(recs['hours'].clip(0)),
    0.0
)

# 3. K-core iterativo (modulo: identico al de H2)
print('K-core filtering...')
recs = iterative_k_core(recs, cfg.min_user, cfg.min_game, verbose=True)
print(f'Post k-core: {recs["user_id"].nunique():,} usuarios | {recs["app_id"].nunique():,} juegos')

# ── Diagnostico FIX #2 (sobre el universo COMPLETO post-k-core) ──
# El muestreador nuevo es invariante al orden de filas; el viejo (pandas.sample)
# NO lo era -> por eso BPR y SBERT/CPGRec evaluaban poblaciones distintas pese a
# usar la misma seed. Mostramos ambos sobre los usuarios reales:
_u      = recs['user_id'].to_numpy()
_u_shuf = np.random.default_rng(0).permutation(_u)
_new_a = set(sample_users(_u,      cfg.frac_eval, cfg.seed))
_new_b = set(sample_users(_u_shuf, cfg.frac_eval, cfg.seed))
_old_a = set(pd.Series(pd.unique(_u)).sample(frac=cfg.frac_eval, random_state=cfg.seed))
_old_b = set(pd.Series(pd.unique(_u_shuf)).sample(frac=cfg.frac_eval, random_state=cfg.seed))
print(f'[FIX#2] nuevo -> pools iguales bajo reordenamiento: {_new_a == _new_b} (|pool|={len(_new_a):,})')
print(f'[FIX#2] viejo -> pools iguales: {_old_a == _old_b} | '
      f'overlap entre dos ordenes = {len(_old_a & _old_b)/max(len(_old_a),1)*100:.1f}%')

# 4-5. Muestreo determinista (FIX#2) + split LOO temporal con politica de
#      pools (modulo): muestrear -> subset train_pool -> LOO -> capear eval.
#      Usa la MISMA seed/fraccion que SBERT -> eval pool identico -> comparable.
sp = build_splits(recs, cfg)
train_df             = sp.train_df
test_pos             = sp.test_pos
test_item_per_user   = sp.test_item_per_user
train_items_per_user = sp.train_items_per_user
eval_users           = sp.eval_users
print(sp.summary())


In [ ]:
user_list   = sorted(train_df['user_id'].unique())
item_list   = sorted(train_df['app_id'].unique())     # catalogo = items en train (consistente con SBERT)
user_to_idx = {u: i for i, u in enumerate(user_list)}
item_to_idx = {a: j for j, a in enumerate(item_list)}
n_users     = len(user_list)
n_items     = len(item_list)

# Interacciones positivas de train: base para matrices ALS y BPR
tp   = positives(train_df, cfg).copy()
tp   = tp[tp['user_id'].isin(user_to_idx) & tp['app_id'].isin(item_to_idx)]
rows = tp['user_id'].map(user_to_idx).to_numpy()
cols = tp['app_id'].map(item_to_idx).to_numpy()

# Popularidad (interacciones positivas/item en train): fallback + novedad
item_pop_dict = popularity_counts(train_df, cfg)
total_inter   = sum(item_pop_dict.values()) or 1
popular_list  = most_popular_list(train_df, cfg)   # desempate determinista (modulo)

eval_users    = sp.eval_users        # ya muestreado/capeado por build_splits
n_train_users = n_users
print(f'n_users={n_users:,} | n_items={n_items:,} | eval_users={len(eval_users):,}')


## 3. Framework de evaluación

Se extiende el H1 de dos formas:
- **Múltiples K**: Precision, Recall y NDCG evaluados en K ∈ {5, 10, 20}.
- **Métricas de diversidad**: Coverage@10, Novelty@10 e ILD@10 (Intra-List Diversity usando factores latentes del modelo).
- **Desagregación por actividad**: NDCG@10 separado por cantidad de interacciones en train del usuario.

In [ ]:
# Las metricas NO se redefinen aqui: vienen de recsys_protocol (importadas en la
# celda de setup). Verificado que son numericamente identicas a las de H2 para
# este pipeline -> evaluate_at_ks / coverage_at_k / ild_at_k / activity_ndcg son
# exactas; novelty_at_k coincide porque todo item recomendado esta en la
# popularidad de train (la unica diferencia teorica es el default para items
# ausentes de pop, que aqui nunca ocurre).
print('Metricas desde recsys_protocol: evaluate_at_ks, coverage_at_k, '
      'novelty_at_k, ild_at_k, activity_ndcg')

## 4. Baseline: Most Popular

Recomienda los juegos con más interacciones positivas en entrenamiento, excluyendo los ya vistos por el usuario. Referencia del H1: NDCG@10 = 0.0222.

In [ ]:
recs_pop = {}
for u in eval_users:
    seen = train_items_per_user.get(u, set())
    recs_pop[u] = [i for i in popular_list if i not in seen][:TOP_K]

metrics_pop = evaluate_at_ks(recs_pop, test_item_per_user)
print('Most Popular')
for k in KS:
    print(f'  P@{k}={metrics_pop[f"Precision@{k}"]:.4f}  '
          f'R@{k}={metrics_pop[f"Recall@{k}"]:.4f}  '
          f'NDCG@{k}={metrics_pop[f"NDCG@{k}"]:.4f}')

## 5. ALS — Filtrado Colaborativo (referencia H1)

ALS con feedback implícito. Minimiza el error cuadrático ponderado por la confianza $c_{ui} = 1 + \alpha \cdot \log(1 + \text{hours})$ con $\alpha=40$. Se incluye para comparación directa con BPR manteniendo los mismos hiperparámetros del H1: `factors=64`, `regularization=0.1`, `iterations=15`.

In [ ]:
conf_data = (1.0 + ALPHA * tp['confidence'].to_numpy()).astype('float32')
user_item = csr_matrix((conf_data, (rows, cols)), shape=(n_users, n_items))

als_model = implicit.als.AlternatingLeastSquares(
    factors=64, regularization=0.1, iterations=15,
    calculate_training_loss=True, random_state=RANDOM_STATE
)
als_model.fit(user_item, show_progress=True)

recs_als = {}
for u in eval_users:
    seen = train_items_per_user.get(u, set())
    if u not in user_to_idx:
        recs_als[u] = [i for i in popular_list if i not in seen][:TOP_K]
        continue
    ui = user_to_idx[u]
    ids, _ = als_model.recommend(
        ui, user_item[ui], N=TOP_K + len(seen), filter_already_liked_items=True
    )
    recs_als[u] = [item_list[j] for j in ids if item_list[j] not in seen][:TOP_K]

metrics_als = evaluate_at_ks(recs_als, test_item_per_user)
div_als = {
    'Coverage@10': coverage_at_k(recs_als, n_items),
    'Novelty@10':  novelty_at_k(recs_als, item_pop_dict, total_inter),
    'ILD@10':      ild_at_k(recs_als, item_to_idx, als_model.item_factors),
}
print('ALS')
for k in KS:
    print(f'  P@{k}={metrics_als[f"Precision@{k}"]:.4f}  '
          f'R@{k}={metrics_als[f"Recall@{k}"]:.4f}  '
          f'NDCG@{k}={metrics_als[f"NDCG@{k}"]:.4f}')
for m, v in div_als.items():
    print(f'  {m}={v:.4f}')

## 6. BPR — Bayesian Personalized Ranking

BPR optimiza directamente el ranking pairwise: para cada usuario $u$, un ítem positivo observado $j$ debe estar mejor rankeado que cualquier ítem no observado $k$:

$$\text{BPR-OPT} = \sum_{(u,j,k) \in D_S} \ln \sigma(\hat{x}_{ujk}) - \lambda \|\Theta\|^2$$

donde $\hat{x}_{ujk} = \hat{x}_{uj} - \hat{x}_{uk}$ es la diferencia de scores y $\sigma$ es la sigmoide.

**Diferencias clave con ALS:**

| | ALS | BPR |
|---|---|---|
| **Objetivo** | Minimiza MSE ponderado por confianza | Maximiza log-verosimilitud del ranking |
| **Señal de entrada** | Matriz de confianza: $1 + 40 \cdot \log(1+h)$ | Matriz binaria 0/1 |
| **Optimización** | Cuadrados mínimos alternados (batch) | SGD sobre triples $(u, j^+, k^-)$ |
| **Alineación top-K** | Indirecta | Directa |

Se usan los mismos `factors=64` que ALS para comparación justa. BPR requiere más iteraciones (100 vs 15) porque SGD converge más lentamente que el método batch de ALS.

In [ ]:
try:
    from implicit.bpr import BayesianPersonalizedRanking
except ImportError:
    from implicit.cpu.bpr import BayesianPersonalizedRanking

# Matriz binaria: BPR optimiza ranking relativo, no intensidad de preferencia
user_item_bpr = csr_matrix(
    (np.ones(len(tp), dtype='float32'), (rows, cols)),
    shape=(n_users, n_items)
)

bpr_model = BayesianPersonalizedRanking(
    factors=64,
    learning_rate=0.01,
    regularization=0.01,
    iterations=100,
    random_state=RANDOM_STATE,
)
bpr_model.fit(user_item_bpr, show_progress=True)

recs_bpr = {}
for u in eval_users:
    seen = train_items_per_user.get(u, set())
    if u not in user_to_idx:
        recs_bpr[u] = [i for i in popular_list if i not in seen][:TOP_K]
        continue
    ui = user_to_idx[u]
    ids, _ = bpr_model.recommend(
        ui, user_item_bpr[ui], N=TOP_K + len(seen), filter_already_liked_items=True
    )
    recs_bpr[u] = [item_list[j] for j in ids if item_list[j] not in seen][:TOP_K]

metrics_bpr = evaluate_at_ks(recs_bpr, test_item_per_user)
div_bpr = {
    'Coverage@10': coverage_at_k(recs_bpr, n_items),
    'Novelty@10':  novelty_at_k(recs_bpr, item_pop_dict, total_inter),
    'ILD@10':      ild_at_k(recs_bpr, item_to_idx, bpr_model.item_factors),
}
print('BPR')
for k in KS:
    print(f'  P@{k}={metrics_bpr[f"Precision@{k}"]:.4f}  '
          f'R@{k}={metrics_bpr[f"Recall@{k}"]:.4f}  '
          f'NDCG@{k}={metrics_bpr[f"NDCG@{k}"]:.4f}')
for m, v in div_bpr.items():
    print(f'  {m}={v:.4f}')

## 7. Comparación de resultados

In [ ]:
# ── Tabla completa de métricas ──
model_info = [
    ('Most Popular', recs_pop, metrics_pop, None),
    ('ALS',          recs_als, metrics_als, als_model.item_factors),
    ('BPR',          recs_bpr, metrics_bpr, bpr_model.item_factors),
]

rows_table = []
for name, recs_m, met, factors in model_info:
    row = {'Modelo': name}
    for k in KS:
        row[f'P@{k}']    = met[f'Precision@{k}']
        row[f'R@{k}']    = met[f'Recall@{k}']
        row[f'NDCG@{k}'] = met[f'NDCG@{k}']
    row['Coverage@10'] = coverage_at_k(recs_m, n_items)
    row['Novelty@10']  = novelty_at_k(recs_m, item_pop_dict, total_inter)
    row['ILD@10']      = ild_at_k(recs_m, item_to_idx, factors) if factors is not None else float('nan')
    rows_table.append(row)

df_results = pd.DataFrame(rows_table).set_index('Modelo')
print('=== Métricas de Ranking y Diversidad ===')
display(df_results.round(4))

# ── Gráfico NDCG@K ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#3498DB', '#E67E22', '#27AE60']
names  = [name for name, *_ in model_info]
for ax, k in zip(axes, KS):
    vals = [met[f'NDCG@{k}'] for _, _, met, _ in model_info]
    bars = ax.bar(names, vals, color=colors, alpha=0.85, edgecolor='black')
    ax.set_title(f'NDCG@{k}', fontsize=13)
    ax.set_ylabel('Score')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0003,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9)
plt.suptitle(f'Comparación de modelos (n={len(eval_users):,} usuarios)', fontsize=13)
plt.tight_layout()
plt.savefig('ndcg_h2_bpr.png', bbox_inches='tight')
plt.show()

# ── Desagregación por nivel de actividad ──
print('\n=== NDCG@10 por nivel de actividad (nº interacciones en train) ===')
act_bins  = ['2-5', '6-10', '11-20', '21+']
act_table = {}
counts    = {}
for name, recs_m, _, _ in model_info:
    act = activity_ndcg(recs_m, test_item_per_user, train_items_per_user)
    act_table[name] = {label: val for label, (val, _) in act.items()}
    if not counts:
        counts = {label: cnt for label, (_, cnt) in act.items()}

df_act = pd.DataFrame(act_table).T[act_bins]
print(f'N usuarios por grupo: {counts}')
display(df_act.round(4))

# ── Mejoras relativas vs Most Popular ──
print('\nMejora de NDCG@10 relativa a Most Popular:')
base = metrics_pop['NDCG@10']
for name, _, met, _ in model_info:
    val   = met['NDCG@10']
    delta = (val / max(base, 1e-9) - 1) * 100
    print(f'  {name:<15}: {val:.4f}  ({delta:+.1f}%)')